In [5]:
import requests
import pandas as pd
import os

In [6]:
# Base URL của API
BASE_URL = "http://api.jolpi.ca/ergast/f1"

# Danh sách endpoint và cấu trúc key_path
endpoints = {
    # Không theo năm
    "season": (["MRData", "SeasonTable", "Seasons"], False),
    "circuit": (["MRData", "CircuitTable", "Circuits"], False),
    "status": (["MRData", "StatusTable", "Status"], False),

    # Theo năm
    "drivers": (["MRData", "DriverTable", "Drivers"], True),
    "constructors": (["MRData", "ConstructorTable", "Constructors"], True),
    "races": (["MRData", "RaceTable", "Races"], True),
    "results": (["MRData", "RaceTable", "Races"], True),
    "driverstandings": (["MRData", "StandingsTable", "StandingsLists"], True),
    "constructorstandings": (["MRData", "StandingsTable", "StandingsLists"], True),
    "qualifying": (["MRData", "RaceTable", "Races"], True),
    "sprint": (["MRData", "RaceTable", "Races"], True),

    # Phụ thuộc năm và vòng đua (ví dụ: "1", "2", ...)
    "pitstop": (["MRData", "RaceTable", "Races"], True),
    "lap": (["MRData", "RaceTable", "Races"], True)
}


In [7]:
# Hàm gọi API
def fetch_data_with_pagination(endpoint, key_path, params=None):
    """
    Hàm hỗ trợ crawl API có phân trang
    - endpoint: đường dẫn endpoint
    - key_path: đường dẫn để trích xuất dữ liệu
    - params: tham số GET (ví dụ: offset)
    """
    all_data = []
    offset = 0
    while True:
        if params is None:
            params = {}
        params["offset"] = offset
        url = f"{BASE_URL}/{endpoint}"
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"Failed to fetch data from {url} with offset {offset}. Status: {response.status_code}")
            break
        
        # Trích xuất dữ liệu
        data = response.json()
        current_data = data
        for key in key_path:
            current_data = current_data.get(key, {})
        
        if isinstance(current_data, list):
            all_data.extend(current_data)
            # Dừng lại nếu không còn dữ liệu
            if len(current_data) < params.get("limit", 30):
                break
        else:
            break

        offset += params.get("limit", 30)
    
    return all_data

In [8]:
# Hàm xử lý API không chia theo năm
def process_static_data(endpoint, key_path, filename):
    """
    Xử lý dữ liệu không phụ thuộc năm (season, circuit, status)
    """
    print(f"Fetching static data from endpoint '{endpoint}'...")
    data = fetch_data_with_pagination(endpoint, key_path)
    if data:
        df = pd.DataFrame(data)
        os.makedirs(f"data1/{endpoint}", exist_ok=True)
        file_path = os.path.join(f"data1/{endpoint}", f"{filename}.csv")
        df.to_csv(file_path, index=False)
        print(f"Saved data for endpoint '{endpoint}' to {file_path}")
    else:
        print(f"No data found for endpoint '{endpoint}'.")

# Hàm xử lý API chia theo năm
def process_yearly_data(endpoint, key_path, year_range=range(2000, 2025)):
    """
    Xử lý dữ liệu phụ thuộc năm (race, driver, constructor,...)
    """
    for year in year_range:
        print(f"Fetching data for endpoint '{endpoint}' in year {year}...")
        year_endpoint = f"{year}/{endpoint}"
        data = fetch_data_with_pagination(year_endpoint, key_path)
        if data:
            df = pd.DataFrame(data)
            os.makedirs(f"data1/{endpoint}", exist_ok=True)
            file_path = os.path.join(f"data1/{endpoint}", f"{endpoint}_{year}.csv")
            df.to_csv(file_path, index=False)
            print(f"Saved data for endpoint '{endpoint}' in year {year} to {file_path}")
        else:
            print(f"No data found for endpoint '{endpoint}' in year {year}.")

def process_round_data(endpoint, key_path, year_range=range(2010, 2025), max_round=22):
    """
    Xử lý dữ liệu phụ thuộc năm và vòng đua (round)
    - endpoint: tên endpoint (vd: pitstop, lap)
    - key_path: đường dẫn để lấy dữ liệu từ JSON
    - year_range: khoảng năm (vd: 2010-2025)
    - max_round: số vòng đua tối đa (tùy thuộc giải đấu)
    """
    for year in year_range:
        for round_number in range(1, max_round + 1):
            print(f"Fetching data for endpoint '{endpoint}' in year {year}, round {round_number}...")
            round_endpoint = f"{year}/{round_number}/{endpoint}"
            data = fetch_data_with_pagination(round_endpoint, key_path)
            if data:
                df = pd.DataFrame(data)
                os.makedirs(f"data/{endpoint}", exist_ok=True)
                file_path = os.path.join(f"data/{endpoint}", f"{endpoint}_{year}_round_{round_number}.csv")
                df.to_csv(file_path, index=False)
                print(f"Saved data for endpoint '{endpoint}' for year {year}, round {round_number} to {file_path}")
            else:
                print(f"No data found for endpoint '{endpoint}' in year {year}, round {round_number}.")


# Xử lý tất cả endpoint
for endpoint, (key_path, is_yearly) in endpoints.items():
    if is_yearly:
        process_yearly_data(endpoint, key_path, year_range=range(2000, 2025))  # Chỉnh year_range tùy ý
    else:
        process_static_data(endpoint, key_path, endpoint)
        
process_round_data("pitstop", ["MRData", "RaceTable", "Races"], year_range=range(2010, 2025), max_round=44)
process_round_data("lap", ["MRData", "RaceTable", "Races"], year_range=range(2010, 2025), max_round=44)

Fetching static data from endpoint 'season'...
Failed to fetch data from http://api.jolpi.ca/ergast/f1/season with offset 0. Status: 400
No data found for endpoint 'season'.
Fetching static data from endpoint 'circuit'...
Failed to fetch data from http://api.jolpi.ca/ergast/f1/circuit with offset 0. Status: 400
No data found for endpoint 'circuit'.
Fetching static data from endpoint 'status'...
Saved data for endpoint 'status' to data1/status\status.csv
Fetching data for endpoint 'drivers' in year 2000...
Saved data for endpoint 'drivers' in year 2000 to data1/drivers\drivers_2000.csv
Fetching data for endpoint 'drivers' in year 2001...
Saved data for endpoint 'drivers' in year 2001 to data1/drivers\drivers_2001.csv
Fetching data for endpoint 'drivers' in year 2002...
Saved data for endpoint 'drivers' in year 2002 to data1/drivers\drivers_2002.csv
Fetching data for endpoint 'drivers' in year 2003...
Saved data for endpoint 'drivers' in year 2003 to data1/drivers\drivers_2003.csv
Fetchi

KeyboardInterrupt: 